In [ ]:
import time
import os
from glob import glob
from pathlib import Path

import torch
import nibabel as nib
import pandas as pd
from natsort import natsorted
from tqdm import tqdm

In [ ]:
root = Path('/media/yannik/Intenso/DATA/dfg_plexus')
pred_dir_hc = root / 'nnUNet/nnUNet_inference_results/Dataset002_DFGFinetuned_3d_fullres_full_new_hc'
pred_dir_ms = root / 'nnUnet/nnUNet_inference_results/Dataset002_DFGFinetuned_3d_fullres_full_ms'

assert pred_dir_ms.exists() and pred_dir_hc.exists()

pred_paths_hc = natsorted(glob(os.path.join(pred_dir_hc.absolute(), '*.nii.gz')))
pred_paths_ms = natsorted(glob(os.path.join(pred_dir_ms.absolute(), '*.nii.gz')))

In [ ]:
print("---------- Loading predicted masks ----------")
time.sleep(0.1)
subjects = []
volumes = []
voxel_size = None  # Variable to hold voxel size for unit determination

for pred_paths, pred_dir, name in [(pred_paths_hc, pred_dir_hc, 'HC'), (pred_paths_ms, pred_dir_ms, 'MS')]:

    for pred_path in tqdm(pred_paths):

        subject = pred_path.split('/')[-1].split('.')[0]
        subjects.append(subject)

        pred_mask = nib.load(pred_path)
        pred_mask_data = torch.from_numpy(pred_mask.get_fdata()).unsqueeze(0).to(torch.float32)

        # Get voxel size from the mask header
        if voxel_size is None:
            voxel_size = pred_mask.header.get_zooms()  # Get the voxel size

        # Compute the volume from positive region
        voxel_volume = torch.prod(torch.tensor(pred_mask.header.get_zooms()))  # Get voxel size
        positive_voxels = torch.sum(pred_mask_data > 0)  # Count positive voxels
        volume = positive_voxels.item() * voxel_volume.item()  # Calculate total volume
        volumes.append(volume)

    # Convert lists to a DataFrame
    volume_data = {'subject_id': subjects, 'volume': volumes}
    df = pd.DataFrame(volume_data)

    # Save the DataFrame to a CSV file
    output_csv_path = root / f"{str(pred_dir.absolute()).split('/')[-1]}___volumes.csv"
    df.to_csv(output_csv_path, index=False)
    print(f"Volumes saved to {output_csv_path}")

    # Calculate and print the mean volume
    mean_volume = sum(volumes) / len(volumes) if volumes else 0
    print(f"Mean volume {name}: {mean_volume:.2f} cubic millimeters (mm³)")

    # Determine the unit based on voxel size
    if voxel_size is not None:
        if all(v == voxel_size[0] for v in voxel_size):  # Check if isotropic
            unit = f"{voxel_size[0]:.2f} mm"  # Assuming mm for isotropic voxels
        else:
            unit = f"({voxel_size[0]:.2f} mm, {voxel_size[1]:.2f} mm, {voxel_size[2]:.2f} mm)"  # List individual dimensions
        print(f"Voxel size {name}: {unit}")
